In [4]:
# ============================================================
# HINDI SENTIMENT ANALYSIS USING MACHINE LEARNING
# ============================================================

# Install required libraries
!pip install datasets indic-nlp-library seaborn wordcloud scikit-learn -q



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from wordcloud import WordCloud


# ============================================================
# 2. LOAD DATASET
# ============================================================

# Corrected to load local CSV file using Hugging Face datasets
dataset = load_dataset("csv", data_files="hindi_sentiment_analysis_dedup.csv")

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment'],
        num_rows: 2500
    })
})


In [6]:


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from wordcloud import WordCloud


# ============================================================
# 2. LOAD DATASET
# ============================================================

# Corrected to load local CSV file using Hugging Face datasets
dataset = load_dataset("csv", data_files="hindi_sentiment_analysis_dedup.csv")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'sentiment'],
        num_rows: 2500
    })
})


In [8]:



# ============================================================
# 3. CONVERT DATASET TO DATAFRAME
# ============================================================

if "train" in dataset:
    df = dataset["train"].to_pandas()
else:
    df = pd.DataFrame(dataset)

print("\nFIRST 5 RECORDS")
print(df.head())

print("\nSHAPE")
print(df.shape)

print("\nCOLUMNS")
print(df.columns)

print("\nDATA TYPES")
print(df.dtypes)


# ============================================================
# 4. CHECK MISSING VALUES
# ============================================================

print("\nMISSING VALUES")
print(df.isnull().sum())


# ============================================================
# 5. CHECK DUPLICATES
# ============================================================

print("\nDUPLICATE RECORDS")
print(df.duplicated().sum())


# ============================================================
# 6. REMOVE MISSING VALUES
# ============================================================

df = df.dropna()

print("\nSHAPE AFTER REMOVING MISSING VALUES")
print(df.shape)


# ============================================================
# 7. REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates()

print("\nSHAPE AFTER REMOVING DUPLICATES")
print(df.shape)


# ============================================================
# 8. CHECK COLUMN NAMES
# ============================================================

print("\nCOLUMN NAMES")
print(df.columns.tolist())


# ============================================================
# 9. IDENTIFY TEXT AND SENTIMENT COLUMNS
# ============================================================

text_column = "text"
label_column = "sentiment"

df[text_column] = df[text_column].astype(str)


# ============================================================
# 10. SENTIMENT LABEL MAPPING
# ============================================================

print("\nUNIQUE SENTIMENT VALUES")
print(df[label_column].unique())


# Convert labels if they are textual
label_mapping = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

if df[label_column].dtype == "object":
    df[label_column] = df[label_column].map(label_mapping)

df = df.dropna(subset=[label_column])

df[label_column] = df[label_column].astype(int)


# ============================================================
# 11. SENTIMENT DISTRIBUTION
# ============================================================

print("\nSENTIMENT DISTRIBUTION")
print(df[label_column].value_counts())


# ============================================================
# 12. PLOT SENTIMENT DISTRIBUTION
# ============================================================

sentiment_names = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

plot_data = df[label_column].map(sentiment_names)

plt.figure(figsize=(7, 5))
sns.countplot(x=plot_data)
plt.title("Hindi Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Records")
plt.show()


# ============================================================
# 13. TEXT PREPROCESSING
# ============================================================

def clean_hindi_text(text):

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Keep Hindi Unicode characters, numbers and spaces
    text = re.sub(r"[^\u0900-\u097F\s0-9]", " ", text)

    # Remove extra spaces again
    text = re.sub(r"\s+", " ", text).strip()

    return text


df["clean_text"] = df[text_column].apply(clean_hindi_text)


# ============================================================
# 14. DISPLAY BEFORE AND AFTER PREPROCESSING
# ============================================================

print("\nBEFORE AND AFTER PREPROCESSING")

print(
    df[[text_column, "clean_text"]].head(10).to_string(index=False)
)


# ============================================================
# 15. IMPORTANT:
#    DO NOT REMOVE NEGATION WORDS
# ============================================================

negation_words = [
    "नहीं",
    "नही",
    "मत",
    "ना",
    "न"
]

print("\nNEGATION WORDS PRESERVED:")
print(negation_words)


# ============================================================
# 16. TEXT LENGTH ANALYSIS
# ============================================================

df["text_length"] = df["clean_text"].apply(
    lambda x: len(x.split())
)

print("\nTEXT LENGTH STATISTICS")
print(df["text_length"].describe())


# ============================================================
# 17. TEXT LENGTH DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(df["text_length"], bins=30)

plt.title("Distribution of Hindi Text Length")
plt.xlabel("Number of Words")
plt.ylabel("Frequency")

plt.show()


# ============================================================
# 18. TRAIN TEST SPLIT
# ============================================================

X = df["clean_text"]
y = df[label_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTRAINING DATA:", len(X_train))
print("TESTING DATA:", len(X_test))


# ============================================================
# 19. TF-IDF REPRESENTATION
# ============================================================

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF TRAINING SHAPE")
print(X_train_tfidf.shape)

print("\nTF-IDF TESTING SHAPE")
print(X_test_tfidf.shape)


# ============================================================
# 20. MODEL 1 - MULTINOMIAL NAIVE BAYES
# ============================================================

nb_model = MultinomialNB()

start_train = time.time()

nb_model.fit(X_train_tfidf, y_train)

nb_train_time = time.time() - start_train


start_prediction = time.time()

nb_pred = nb_model.predict(X_test_tfidf)

nb_prediction_time = time.time() - start_prediction


# ============================================================
# 21. MODEL 2 - LOGISTIC REGRESSION
# ============================================================

lr_model = LogisticRegression(
    max_iter=1000
)

start_train = time.time()

lr_model.fit(X_train_tfidf, y_train)

lr_train_time = time.time() - start_train


start_prediction = time.time()

lr_pred = lr_model.predict(X_test_tfidf)

lr_prediction_time = time.time() - start_prediction


# ============================================================
# 22. MODEL 3 - LINEAR SVM
# ============================================================

svm_model = LinearSVC()

start_train = time.time()

svm_model.fit(X_train_tfidf, y_train)

svm_train_time = time.time() - start_train


start_prediction = time.time()

svm_pred = svm_model.predict(X_test_tfidf)

svm_prediction_time = time.time() - start_prediction


# ============================================================
# 23. EVALUATION FUNCTION
# ============================================================

def evaluate_model(model_name, y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    print("\n========================================")
    print(model_name)
    print("========================================")

    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

    print("\nClassification Report")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Negative",
                "Neutral",
                "Positive"
            ],
            zero_division=0
        )
    )

    return accuracy, precision, recall, f1


# ============================================================
# 24. EVALUATE ALL MODELS
# ============================================================

nb_scores = evaluate_model(
    "Multinomial Naive Bayes",
    y_test,
    nb_pred
)

lr_scores = evaluate_model(
    "Logistic Regression",
    y_test,
    lr_pred
)

svm_scores = evaluate_model(
    "Linear SVM",
    y_test,
    svm_pred
)


# ============================================================
# 25. MODEL COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame({

    "Model": [
        "Multinomial Naive Bayes",
        "Logistic Regression",
        "Linear SVM"
    ],

    "Accuracy": [
        nb_scores[0],
        lr_scores[0],
        svm_scores[0]
    ],

    "Precision": [
        nb_scores[1],
        lr_scores[1],
        svm_scores[1]
    ],

    "Recall": [
        nb_scores[2],
        lr_scores[2],
        svm_scores[2]
    ],

    "F1 Score": [
        nb_scores[3],
        lr_scores[3],
        svm_scores[3]
    ],

    "Training Time (sec)": [
        nb_train_time,
        lr_train_time,
        svm_train_time
    ],

    "Prediction Time (sec)": [
        nb_prediction_time,
        lr_prediction_time,
        svm_prediction_time
    ]
})


print("\nMODEL COMPARISON")
print(comparison)


# ============================================================
# 26. ACCURACY GRAPH
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    comparison["Model"],
    comparison["Accuracy"]
)

plt.title("Model Accuracy Comparison")
plt.xlabel("Model")
plt.ylabel("Accuracy")

plt.xticks(rotation=20)

plt.ylim(0, 1)

plt.show()


# ============================================================
# 27. F1 SCORE GRAPH
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    comparison["Model"],
    comparison["F1 Score"]
)

plt.title("Model F1 Score Comparison")
plt.xlabel("Model")
plt.ylabel("Macro F1 Score")

plt.xticks(rotation=20)

plt.ylim(0, 1)

plt.show()


# ============================================================
# 28. CONFUSION MATRIX - NAIVE BAYES
# ============================================================

cm_nb = confusion_matrix(y_test, nb_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_nb,
    annot=True,
    fmt="d",
    xticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    yticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

plt.title("Confusion Matrix - Naive Bayes")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ============================================================
# 29. CONFUSION MATRIX - LOGISTIC REGRESSION
# ============================================================

cm_lr = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    xticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    yticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ============================================================
# 30. CONFUSION MATRIX - LINEAR SVM
# ============================================================

cm_svm = confusion_matrix(y_test, svm_pred)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_svm,
    annot=True,
    fmt="d",
    xticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ],
    yticklabels=[
        "Negative",
        "Neutral",
        "Positive"
    ]
)

plt.title("Confusion Matrix - Linear SVM")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ============================================================
# 31. WORD CLOUD
# ============================================================

all_text = " ".join(df["clean_text"])

wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    font_path=None
).generate(all_text)

plt.figure(figsize=(12, 6))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Hindi Sentiment Dataset Word Cloud")

plt.show()


# ============================================================
# 32. TEST CUSTOM HINDI SENTENCES
# ============================================================

def predict_sentiment(text):

    cleaned = clean_hindi_text(text)

    vector = tfidf.transform([cleaned])

    prediction = svm_model.predict(vector)[0]

    return sentiment_names[prediction]


test_sentences = [
    "यह फिल्म बहुत अच्छी है।",
    "यह फिल्म बिल्कुल अच्छी नहीं है।",
    "फिल्म ठीक ठाक है।",
    "मुझे यह बहुत पसंद आया।",
    "यह बहुत खराब अनुभव था।"
]

print("\nCUSTOM SENTIMENT PREDICTIONS")

for sentence in test_sentences:

    result = predict_sentiment(sentence)

    print("\nText:", sentence)
    print("Sentiment:", result)


# ============================================================
# 33. SAVE MODEL AND TF-IDF VECTORIZER
# ============================================================

import joblib

joblib.dump(
    svm_model,
    "hindi_sentiment_svm.pkl"
)

joblib.dump(
    tfidf,
    "hindi_sentiment_tfidf.pkl"
)

print("\nModel and TF-IDF vectorizer saved successfully.")


# ============================================================
# 34. FINAL RESULT
# ============================================================

best_model = comparison.loc[
    comparison["F1 Score"].idxmax()
]

print("\n========================================")
print("BEST MODEL")
print("========================================")

print("Model:", best_model["Model"])

print(
    "Accuracy:",
    round(best_model["Accuracy"], 4)
)

print(
    "Macro F1:",
    round(best_model["F1 Score"], 4)
)


FIRST 5 RECORDS
                              sentence sentiment
0           आज का मौसम बहुत सुहाना है।  positive
1  मुझे यह खाना बिल्कुल पसंद नहीं आया।  negative
2     मेरा इस विषय पर कोई राय नहीं है।   neutral
3      आपकी मदद के लिए दिल से धन्यवाद।  positive
4         यह अनुभव बेहद निराशाजनक रहा।  negative

SHAPE
(2500, 2)

COLUMNS
Index(['sentence', 'sentiment'], dtype='str')

DATA TYPES
sentence     str
sentiment    str
dtype: object

MISSING VALUES
sentence     0
sentiment    0
dtype: int64

DUPLICATE RECORDS
0

SHAPE AFTER REMOVING MISSING VALUES
(2500, 2)

SHAPE AFTER REMOVING DUPLICATES
(2500, 2)

COLUMN NAMES
['sentence', 'sentiment']


KeyError: 'text'